# Module 08: Computational Research Design, Paper Audits & Validation Lab

**Core Validation Pipeline:** $\text{Question} \to \text{Model & Null Baseline} \to \text{Sensitivity & BH-FDR Test} \to \text{Defensible Claim}$
- **Track A (BH-FDR Power Calibration):** Solve minimum sample size $T_{80}$ for $80\%$ detection power under Benjamini–Hochberg FDR control.
- **Track B (Null Baseline & Effect Size):** Standardized separation Cohen's $d = \frac{\mu_{\text{mech}} - \mu_{\text{null}}}{s_{\text{pooled}}}$.
- **8-Question Paper Audit Protocol & 5-Part Research Brief Builder** across all 6 workshop classics.

In [ ]:
import subprocess
import sys
from pathlib import Path

def _ensure_toolkit_on_path():
    cwd = Path.cwd().resolve()
    for cand in (cwd, cwd.parent, cwd.parent.parent):
        if (cand / "core").is_dir():
            if str(cand) not in sys.path:
                sys.path.insert(0, str(cand))
            return cand
    clone_dir = cwd / "The-Collective-Intelligence-Toolkit"
    if not clone_dir.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/poria-azadi96/The-Collective-Intelligence-Toolkit.git", str(clone_dir)],
            check=True,
        )
    if str(clone_dir) not in sys.path:
        sys.path.insert(0, str(clone_dir))
    return clone_dir

REPO_ROOT = _ensure_toolkit_on_path()
import matplotlib.pyplot as plt
import ipywidgets as widgets
from core.research_validation import (
    PAPER_AUDITS,
    build_research_brief,
    create_module08_dashboard_figure,
    run_module08_tier1,
    run_module08_tier2,
    run_module08_tier3,
    simulate_fdr_validation as _core_sim_fdr,
)
from core.real_world_demos_part2 import (
    demo_m08_grand_synthesis_pipeline,
    demo_m08_industrial_rd_power_pipeline,
    demo_m08_q1_peer_review_stress_test,
)

def simulate_fdr_validation(n_tests=50, true_edges=8, signal_snr=0.55, sample_size=300, seed=42):
    res = _core_sim_fdr(n_tests=n_tests, true_edges=true_edges, signal_snr=signal_snr, sample_size=sample_size, n_mc=60, seed=seed)
    return (res['raw_tpr'], res['raw_fdp']), (res['fdr_tpr'], res['fdr_fdp'])

def run_module08_lab(
    paper_choice='1. Schreiber (2000) - Transfer Entropy',
    track='Track A: Observational FDR & Power',
    sample_size=300,
    alpha=0.05,
):
    fig, stats = create_module08_dashboard_figure(
        paper_choice=paper_choice,
        track=track,
        sample_size=sample_size,
        alpha=alpha,
    )
    plt.show()
    audit = PAPER_AUDITS[paper_choice]
    print('=' * 88)
    print(f'8-QUESTION PAPER AUDIT PROTOCOL | {paper_choice}')
    print('=' * 88)
    for line in audit['setup'] + audit['critique']:
        print(line)
    print('=' * 88)
    print('Validation Metrics:', {k: v for k, v in stats.items() if k != 'calibration'})

widgets.interact(
    run_module08_lab,
    paper_choice=widgets.Dropdown(options=list(PAPER_AUDITS.keys()), value='1. Schreiber (2000) - Transfer Entropy', description='Paper Audit:'),
    track=widgets.Dropdown(options=['Track A: Observational FDR & Power', 'Track B: Generative ABM vs Null Model'], value='Track A: Observational FDR & Power', description='Validation:'),
    sample_size=widgets.IntSlider(min=100, max=1200, step=100, value=300, description='Sample T:'),
    alpha=widgets.Dropdown(options=[0.01, 0.05, 0.10], value=0.05, description='FDR alpha:'),
);

## 3-Tier Experimental Protocol
- **Tier 1 (Paper Audit Inspection):** Visualize how Q7 (Limitations) directly generates Q8 (Open Research Question) across all 6 Classics.
- **Tier 2 (Track A Calibration):** Solve minimum sample size $T_{80}$ reaching $80\%$ power under BH-FDR.
- **Tier 3 (Track B Null Baseline & Research Brief):** Verify Cohen's $d$ separation and generate a 1-page 5-Part Research Brief.

In [ ]:
def run_m08_tier_selector(tier='Tier 1: Paper Audit Q7 -> Q8 Matrix'):
    if tier.startswith('Tier 1'):
        fig, res = run_module08_tier1()
    elif tier.startswith('Tier 2'):
        fig, res = run_module08_tier2()
    else:
        fig, res = run_module08_tier3()
        print(res['sample_brief']['markdown'])
        res = {'focused_cohen_d': res['focused_cohen_d'], 'max_cohen_d': res['max_cohen_d']}
    plt.show()
    print(res)

widgets.interact(
    run_m08_tier_selector,
    tier=widgets.Dropdown(
        options=[
            'Tier 1: Paper Audit Q7 -> Q8 Matrix',
            'Tier 2: Track A Power Calibration (T_80)',
            'Tier 3: Track B Null Baseline & Research Brief',
        ],
        value='Tier 1: Paper Audit Q7 -> Q8 Matrix',
        description='Protocol:',
    ),
);

## Module 08 Real-World Applications
1. **Q1 Peer-Review Stress-Testing Suite:** Reproducing Q7 failure modes across all workshop models.
2. **Industrial R&D Power & Null-Model Simulator:** Pre-experiment $T_{80}$ & Cohen's $d$ validation pipeline.
3. **Cross-Module Grand Synthesis Report:** Unified reproducibility scorecard across Modules 01–07.

In [ ]:
def run_m08_real_world_demo(application='1. Q1 Peer-Review Stress-Test Suite'):
    if application.startswith('1.'):
        fig, res = demo_m08_q1_peer_review_stress_test()
    elif application.startswith('2.'):
        fig, res = demo_m08_industrial_rd_power_pipeline()
    else:
        fig, res = demo_m08_grand_synthesis_pipeline()
        res = {'pillar_scores': res['pillar_scores']}
    plt.show()
    print(res)

widgets.interact(
    run_m08_real_world_demo,
    application=widgets.Dropdown(
        options=[
            '1. Q1 Peer-Review Stress-Test Suite',
            '2. Industrial R&D Power & Null Simulator',
            '3. Cross-Module Grand Synthesis Pipeline',
        ],
        value='1. Q1 Peer-Review Stress-Test Suite',
        description='Case Study:',
    ),
);